In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import accuracy_score, classification_report
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

# LOAD DATASET
dataset = Planetoid(root='./data', name='Cora')
data = dataset[0]

print(data)
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Number of features:", dataset.num_features)
print("Number of classes:", dataset.num_classes)

# VISUALIZE GRAPH
G = nx.karate_club_graph()
plt.figure(figsize=(8,6))
nx.draw(G, node_color='lightblue', with_labels=True, node_size=500)
plt.title("Sample Graph Visualization")
plt.show()

# DEFINE GCN MODEL
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# CREATE MODEL
model = GCN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# TRAIN MODEL
for epoch in range(20):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    print("Epoch:", epoch, "Loss:", loss.item())

# TEST MODEL
model.eval()
pred = model(data).argmax(dim=1)
acc = accuracy_score(data.y.cpu(), pred.cpu())
print("Accuracy:", acc)

print("\nClassification Report")
print(classification_report(data.y.cpu(), pred.cpu()))

# PREDICT SINGLE NODE
node_id = int(input("Enter node id (0-2707): "))
print("Predicted class:", pred[node_id].item())

# SAVE MODEL
torch.save(model.state_dict(), "gnn_model.pth")